In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("merged_university_rankings.csv")
print("Duplicates before:", df.duplicated().sum())
print("Shape of dataset:", df.shape)

Duplicates before: 0
Shape of dataset: (1504, 43)


In [3]:
print("Blank university names:", df['Name'].isnull().sum())
print("Empty university names:", (df['Name']=="").sum())

Blank university names: 0
Empty university names: 0


In [4]:
df["Country_Standard"]=df["Country/Territory"].str.strip()

In [5]:
missing=df.isnull().sum()
missing=missing[missing>0]
print(missing)

Previous Rank                             115
Size                                        1
Research                                    1
Status                                     48
International Faculty  SCORE               87
International Faculty  RANK                87
International Student SCORE                37
International student  RANK                37
International Students Diversity SCORE     37
International Students Diversity RANK      37
International Research Network SCORE        2
International Research Network RANK         2
Sustainability SCORE                       24
Sustainability RANK                        24
Rank_THE                                  969
Country                                   969
Student Population                        969
Students to Staff Ratio                   969
International Students                    969
Female to Male Ratio                      987
Overall Score                             969
Teaching                          

In [6]:
for col in ["Size", "Research", "Status"]:
    df[col]=df[col].fillna(df[col].mode()[0])

In [7]:
score_cols=["International Faculty  SCORE", "International Student SCORE",
            "International Students Diversity SCORE", "International Research Network SCORE",
            "Sustainability SCORE"]
for col in score_cols:
    df[col]=df[col].fillna(df[col].median())

In [8]:
missing=df.isnull().sum()
missing=missing[missing>0]
print(missing)

Previous Rank                            115
International Faculty  RANK               87
International student  RANK               37
International Students Diversity RANK     37
International Research Network RANK        2
Sustainability RANK                       24
Rank_THE                                 969
Country                                  969
Student Population                       969
Students to Staff Ratio                  969
International Students                   969
Female to Male Ratio                     987
Overall Score                            969
Teaching                                 969
Research Environment                     969
Research Quality                         969
Industry Impact                          969
International Outlook                    969
Year                                     969
dtype: int64


In [9]:
def normalize_rank(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    value = value.replace("=", "")
    if "-" in value:
        start, end = value.split("-")
        return (float(start) + float(end)) / 2
    return float(value)

df["QS_Rank_Normalized"] = df["Rank_QS"].apply(normalize_rank)
df["THE_Rank_Normalized"] = pd.to_numeric(df["Rank_THE"],errors="coerce")

print(df[["Rank_QS","QS_Rank_Normalized","Rank_THE","THE_Rank_Normalized"]].head(10))

  Rank_QS  QS_Rank_Normalized  Rank_THE  THE_Rank_Normalized
0       1                 1.0       NaN                  NaN
1       2                 2.0       8.0                  8.0
2       3                 3.0       NaN                  NaN
3       4                 4.0       1.0                  1.0
4       5                 5.0       NaN                  NaN
5       6                 6.0       4.0                  4.0
6       7                 7.0       NaN                  NaN
7       8                 8.0       NaN                  NaN
8       9                 9.0       NaN                  NaN
9      10                10.0       NaN                  NaN


In [10]:
df["Sustainability RANK"]=(df["Sustainability RANK"].apply(normalize_rank))

In [11]:
def clean_female_male_ratio(x):
    if pd.isna(x) or x == "":
        return np.nan
    x = str(x).strip()
    # if format= 37 : 63
    if ":" in x and "day" not in x:
        female, male = x.split(":")
        return float(female.strip())/float(male.strip())
    # if format= 2 days, 4:48:00
    if "day" in x:
        td = pd.to_timedelta(x)
        return td.total_seconds()/86400
    return pd.to_numeric(x, errors="coerce")

df["Female to Male Ratio"] = df["Female to Male Ratio"].apply(clean_female_male_ratio)

In [12]:
df.drop(columns=["Country","Country/Territory"], axis=1, inplace=True)

In [13]:
df = df.rename(columns={
    "Previous Rank": "Previous_Rank",
    "Academic Reputation SCORE": "Academic_reputation_score",
    "Academic Reputation RANK": "Academic_Reputation_Rank",
    "Employer Reputation SCORE": "Employer_Reputation_Score",
    "Employer Reputation RANK": "Employer_Reputation_Rank",
    "Faculty Student Ratio SCORE": "Faculty_Student_Ratio_Score",
    "Faculty Student Ratio RANK": "Faculty_Student_Ratio_Rank",
    "Citations per Faculty SCORE": "Citations_per_Faculty_Score",
    "Citations per Faculty RANK": "Citations_per_Faculty_Rank",
    "International Faculty SCORE": "International_Faculty_Score",
    "International Faculty RANK": "International_Faculty_Rank",
    "International Student SCORE": "International_Student_Score",
    "International student RANK": "International_Student_Rank",
    "International Students Diversity SCORE": "International_Students_Diversity_Score",
    "International Students Diversity RANK": "International_Students_Diversity_Rank",
    "International Research Network SCORE": "International_Research_Network_Score",
    "International Research Network RANK": "International_Research_Network_Rank",
    "Employment Outcomes SCORE": "Employment_Outcomes_Score",
    "Employment Outcomes RANK": "Employment_Outcomes_Rank",
    "Sustainability SCORE": "Sustainability_Score",
    "Sustainability RANK": "Sustainability_Rank",
    "Overall SCORE": "QS_Overall_Score",
    "Student Population": "Student_Population",
    "Students to Staff Ratio": "Students_to_Staff_Ratio",
    "International Students": "International_Students",
    "Female to Male Ratio": "Female_to_Male_Ratio",
    "Overall Score": "THE_Overall_Score",
    "Research Environment": "Research_Environment",
    "Research Quality": "Research_Quality",
    "Industry Impact": "Industry_Impact",
    "International Outlook": "International_Outlook",
    "Country_Standard": "Country"
})

In [14]:
total_cells=df.shape[0]*df.shape[1]
missing_cells=df.isnull().sum().sum()
missing_percentage=(missing_cells/total_cells)*100
print("Total cells:", total_cells)
print("Missing cells:", missing_cells)
print("Missing percentage:", missing_percentage)

Total cells: 66176
Missing cells: 12917
Missing percentage: 19.519161025145067


In [15]:
df=df[df['Rank_THE'].notna()].copy()

In [16]:
total_cells=df.shape[0]*df.shape[1]
missing_cells=df.isnull().sum().sum()
missing_percentage=(missing_cells/total_cells)*100
print("Total cells:", total_cells)
print("Missing cells:", missing_cells)
print("Missing percentage:", missing_percentage)

Total cells: 23540
Missing cells: 103
Missing percentage: 0.43755310110450296


In [17]:
print(df.dtypes)

Rank_QS                                    object
Previous_Rank                              object
Name                                       object
Region                                     object
Size                                       object
Focus                                      object
Research                                   object
Status                                     object
Academic_reputation_score                 float64
Academic Reputation  RANK                   int64
Employer_Reputation_Score                 float64
Employer_Reputation_Rank                    int64
Faculty_Student_Ratio_Score               float64
Faculty_Student_Ratio_Rank                  int64
Citations_per_Faculty_Score               float64
Citations_per_Faculty_Rank                  int64
International Faculty  SCORE              float64
International Faculty  RANK                object
International_Student_Score               float64
International student  RANK               float64


In [18]:
df["QS_Overall_Score"]=pd.to_numeric(df["QS_Overall_Score"],errors="coerce")

In [19]:
df["Female_to_Male_Ratio"]=df["Female_to_Male_Ratio"].round(2)
df["THE_Overall_Score"]=df["THE_Overall_Score"].round(2)

In [20]:
df.to_csv("cleaned_university_rankings.csv", index=False)

In [21]:
print(df.shape)

(535, 44)
